# colab6 — Design B: weights encode Y directly (`relu(w_i)`, no Y in forward pass)

## Motivation

In the original paper's architecture, the trainable layer computes `relu(w_i * y[i])`. This is architecturally circular: Y is fed as input, and the weight just learns to be a transparent scaler (`w_i = 1`). The weight doesn't encode Y — the input does.

**Design B** replaces this with `relu(w_i)` — a constant `1` goes into the Dense layer. Now the weight **is** the y-character. The network only receives X as input and must discover Y from edit-distance supervision alone.

## What we tried first (and why it failed)

We first tested Design B with the `{1, 2}` relabeled alphabet (from colab5) and weight initialization in `[0.1, 2.5]`. **Result: `w_2` ran away to ~3 instead of converging to its target.** The cause: the frozen `matching_module` was analytically derived for inputs where `|a - b| ∈ {0, 1}`. When `relu(w_i) > 2`, the matching module saturates (always outputs 1 for all x), gradients die, and the optimizer gets trapped in a false plateau at `loss ≈ 0.5`.

## The fix: go back to `{0, 1}` — but with Design B

The key insight: **Design B removes the `y[i]` factor from the chain rule entirely:**

```
Design A (paper):  d(relu(w_i * y[i]))/dw_i = relu'(...) * y[i]   ← y[i]=0 kills gradient
Design B (ours):   d(relu(w_i))/dw_i        = relu'(w_i)          ← no y[i] factor
```

So the original dead-gradient problem (`y[i] = 0` multiplied into the chain rule) **doesn't exist in Design B**. The `{1, 2}` relabeling was necessary for Design A but is unnecessary here.

With `{0, 1}` encoding, Design B, and initialization in `[0, 1]`:

- `y[i] = 1` → target is `relu(w_i) = 1` → `w_i = 1`. Gradient = `relu'(w_i) = 1` throughout. Clean convergence.
- `y[i] = 0` → target is `relu(w_i) = 0` → `w_i ≤ 0`. If `w_i` starts positive, `relu'(w_i) = 1` so gradient is alive. SGD pushes `w_i` down to 0. Once there, `relu(0) = 0` (correct) and gradient naturally goes to 0 (we're at the target — no need to move further).
- Init in `[0, 1]` keeps `relu(w_i) ∈ [0, 1]` — exactly the `matching_module`'s designed operating range. No false plateaus.

Expected final weights: the **actual character values** of the target string. `[0, 1]` for DESIRE='01', `[1, 1]` for DESIRE='11', `[0, 1, 0, 1, 1]` for DESIRE='01011'.

## Setup

In [ ]:
import os

!rm -rf /content/thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git

os.chdir('/content/thesis-edit-distance-nn')

!ls sampledata/

In [ ]:
!pip install tensorflow ml_dtypes --upgrade

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import random
import time
import os
import math
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Activation, Input, add, Lambda, Reshape, concatenate, Flatten

## Architecture

All frozen modules (`matching_module`, `min_module`, `minimum`) are identical to the original paper.

The **only** change is in `align_model_for_N`: the `for_gen_dense_*` loop feeds a constant `1` into the trainable Dense layer instead of `y[i]` from the input.

In [ ]:
def transform_seqs_to_input(seqA, seqB):
    matching_pairs = []
    input_length_x = 0

    matching_pairs.append([int(seqA[0]), int(seqB[0])])
    if len(seqA) == 1 and len(seqB) == 1:
        return matching_pairs
    else:
        input_length_x = len(seqA)
        match_layers_i = (input_length_x * 2) - 1

    start_i = 1
    end_i = 2

    for l in range(match_layers_i):
        if l < input_length_x - 1:
            i, j = [*reversed(range(0, end_i))], [*range(0, end_i)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            end_i += 1
        else:
            i, j = [*reversed(range(start_i, input_length_x))], [*range(start_i, input_length_x)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            start_i += 1
            if start_i > len(seqB):
                break

    return matching_pairs


def transform_input_for_generate(input):
    x = []
    y = []
    for pair in input:
        x.append(pair[0])
        y.append(pair[1])
    return [x, y]

In [ ]:
def matching_module():
    epsilon = 1

    model = Sequential()
    model.add(Dense(units=2, activation='relu', use_bias=True, input_shape=(2,)))
    model.add(Dense(units=2, activation='relu', use_bias=True))
    model.add(Dense(units=1, activation='relu', use_bias=True))

    w1 = model.layers[0].get_weights()
    w1[0][0][0], w1[0][0][1] = 1.0, -1.0
    w1[0][1][0], w1[0][1][1] = -1.0, 1.0
    w1[1][0], w1[1][1] = 0, 0
    w2 = model.layers[1].get_weights()
    w2[0][0][0], w2[0][0][1] = 1.0, 1.0
    w2[0][1][0], w2[0][1][1] = 1.0, 1.0
    w2[1][0], w2[1][1] = epsilon, -1 * epsilon
    w3 = model.layers[2].get_weights()
    w3[0][0][0], w3[0][1][0] = (1.0/epsilon), -1.0 * (1.0/epsilon)
    w3[1][0] = -1

    model.layers[0].set_weights(w1)
    model.layers[1].set_weights(w2)
    model.layers[2].set_weights(w3)

    model.trainable = False
    return model

In [ ]:
def min_module(i, j, k):
    input = Input(shape=(2,))
    x = Dense(2, activation='relu', use_bias=True)(input)
    combined = concatenate([x, input])

    layer_name = 'result_pixel_' + str(i) + str(j) + '_' + str(k)
    z = Dense(1, activation='relu', use_bias=True, name=layer_name)(combined)
    model = Model(inputs=input, outputs=z)

    w1 = model.layers[1].get_weights()
    w1[0][0], w1[0][1] = [-1.0, 1.0], [1.0, -1.0]
    w2 = model.layers[3].get_weights()
    w2[0][0], w2[0][1], w2[0][2], w2[0][3] = -0.5, -0.5, 0.5, 0.5

    model.layers[1].set_weights(w1)
    model.layers[3].set_weights(w2)

    model.trainable = False
    return model


def minimum(i, j):
    input = Input(shape=(3,))
    comp1_pair = Lambda(lambda x: x[:, :2], output_shape=(2,))(input)
    comp2_input = Lambda(lambda x: x[:, 2:], output_shape=(1,))(input)

    m = min_module(i, j, 1)(comp1_pair)
    comp2_pair = concatenate([comp2_input, m])
    output = min_module(i, j, 2)(comp2_pair)

    model = Model(inputs=input, outputs=output)
    model.trainable = False
    return model

In [ ]:
def align_model_for_N(seq_length_x, seq_length_y, matching_pair_number):
    input = Input(shape=(2, matching_pair_number), name='input')

    y = Lambda(lambda t: t[:, 1, :], output_shape=(matching_pair_number,))(input)
    x = Lambda(lambda t: t[:, 0, :], output_shape=(matching_pair_number,))(input)

    out = {}
    start_i = 0
    step = 2
    for i in range(seq_length_y):
        a = start_i
        layername = 'for_gen_dense_' + str(i + 1)
        # === DESIGN B ===
        # Feed constant 1 instead of y[i]. Output = relu(w_i * 1) = relu(w_i).
        # The weight IS the learned y-character. No y in the forward pass.
        ones_input = Lambda(lambda t, a=a: tf.ones_like(t[:, a:a+1]), output_shape=(1,))(y)
        z = Dense(1, activation='relu', name=layername, use_bias=False)(ones_input)
        out[layername] = z
        start_i += step
        step += 1

    pair_i = 1
    calc_layer = (seq_length_x * 2) - 1
    test_dict = {}

    y_dense_layer_name = 'for_gen_dense_1'
    densed_y = out[y_dense_layer_name]
    x_char = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(x)

    debug_name = 'matching_debug_1'
    pair_11 = concatenate([x_char, densed_y], name=debug_name)

    ext_gaps = Dense(2, activation='relu', name='first_calc_gap_layer')(pair_11)

    min1 = min_module(1, 1, 1)(ext_gaps)
    matching1 = matching_module()(pair_11)
    combined = concatenate([min1, matching1])
    z = min_module(1, 1, 2)(combined)
    result_pixel_11 = concatenate([ext_gaps, z], name='input_pixel_1_1')

    pair_i = 2

    if seq_length_x == 1 and seq_length_y == 1:
        output = z
        return Model(inputs=input, outputs=output)
    else:
        test_dict['input_pixel_1_1'] = result_pixel_11
        test_dict['result_pixel_1_1'] = z

        comp_i_val, comp_j_val = 1, 2
        start_sentinel, end_sentinel = 1, 2
        unbalance_flag = True

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_length_x - 1:
                comp_i_val, comp_j_val = start_sentinel, end_sentinel
                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_i = comp_i_val
                        y_dense_layer_name = 'for_gen_dense_' + str(y_i)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)

                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        if comp_i_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        elif comp_j_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        else:
                            previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                            previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                            previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                            before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                if end_sentinel + 1 <= seq_length_x:
                    end_sentinel += 1

            else:
                start_sentinel += 1
                comp_i_val, comp_j_val = start_sentinel, end_sentinel

                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_dense_layer_name = 'for_gen_dense_' + str(comp_i_val)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)
                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                        previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                        previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                        before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                    if start_sentinel == end_sentinel:
                        return Model(inputs=input, outputs=result_pixel)

In [ ]:
def set_weight_for_debug(model, seq_len_x, seq_len_y, matching_pair):
    """Set analytical weights on frozen layers. Trainable for_gen_dense weights remain at random init."""
    print('setting frozen weights ...')

    w = model.get_layer('first_calc_gap_layer').get_weights()
    w[0][0][0], w[0][0][1] = 0, 0
    w[0][1][0], w[0][1][1] = 0, 0
    w[1][0], w[1][1] = 2, 2
    model.get_layer('first_calc_gap_layer').set_weights(w)
    model.get_layer('first_calc_gap_layer').trainable = False

    if seq_len_x > 1:
        calc_layer = (seq_len_x * 2) - 1
        comp_i, comp_j = 1, 2
        start_sentinel, end_sentinel = 1, 2

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_len_x - 1:
                comp_i, comp_j = start_sentinel, end_sentinel
                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        if comp_i == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        elif comp_j == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        else:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, 0

                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)
                if end_sentinel + 1 <= seq_len_x:
                    end_sentinel += 1
            else:
                start_sentinel = start_sentinel + 1
                comp_i, comp_j = start_sentinel, end_sentinel

                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                        w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                        w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                        w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                        w[1][0], w[1][1], w[1][2] = 1, 1, 0
                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)


def froozen_align_model(model):
    print('freezing all non-trainable layers ...')
    for layer in model.layers:
        if 'for_gen_dense' in layer.name:
            layer.trainable = True
        else:
            layer.trainable = False

## Generate length-5 CSV for DESIRE='01011'

The length-2 CSVs already exist in `sampledata/`. For length 5, we generate all 32 binary strings and their edit distances to `'01011'`.

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

target_01011 = '01011'
csv_path_01011 = '/content/thesis-edit-distance-nn/sampledata/desired_length_5_levenshtein_01011.csv'

lines_01011 = []
for i in range(32):
    s = format(i, '05b')
    d = levenshtein(s, target_01011)
    lines_01011.append(f'{s},{target_01011},{d}')

with open(csv_path_01011, 'w') as f:
    f.write('\n'.join(lines_01011) + '\n')

print(f'wrote {len(lines_01011)} rows')
for line in lines_01011[:5]:
    print(' ', line)

## SGD training (Design B, `{0, 1}` alphabet, init in `[0, 1]`)

Same vanilla SGD loop. Key settings:

- **Alphabet:** original `{0, 1}` — no relabeling needed since Design B eliminates the `y[i]` factor from the chain rule
- **Init range:** `[0, 1]` — keeps `relu(w_i) ∈ [0, 1]`, inside `matching_module`'s designed operating range
- **Expected weights:** the actual character values of the target string

In [ ]:
def training(desire, csv_path, epochs=20, learning_rate=0.1, seed=None):
    """
    Vanilla SGD training. In Design B, weights should converge to the
    actual character values of `desire`.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)

    LEN = len(desire)

    lines = []
    with open(csv_path, 'r') as f:
        for line in f:
            lines.append(line.rstrip('\n'))

    # Verify model correctness with oracle weights before training
    target_weights = [int(c) for c in desire]
    print(f'=== DESIRE={desire} ===')
    print(f'target weights: {target_weights}')
    print(f'training samples: {len(lines)}')

    sp = lines[0].split(',')
    x, y = sp[0], sp[1]
    pairs = transform_seqs_to_input(x, y)
    SEQ_LEN_X = len(x)
    SEQ_LEN_Y = len(y)
    PAIRS_LEN = len(pairs)

    model = align_model_for_N(SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    set_weight_for_debug(model, SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    froozen_align_model(model)

    # Verify: set oracle weights and check all predictions are correct
    print('\nVerify: setting oracle weights to target...')
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        w = model.get_layer(lname).get_weights()
        w[0][0][0] = float(target_weights[i])
        model.get_layer(lname).set_weights(w)

    all_correct = True
    for line in lines:
        sp = line.split(',')
        xs, ys, true_score = sp[0], sp[1], int(sp[2])
        inp = transform_seqs_to_input(xs, ys)
        inp = transform_input_for_generate(inp)
        inp = tf.constant([inp], dtype=tf.float32)
        pred = float(model(inp, training=False)[0][0])
        if abs(pred - true_score) > 0.01:
            print(f'  MISMATCH: x={xs} y={ys} true={true_score} pred={pred:.4f}')
            all_correct = False
    print(f'  {"ALL CORRECT" if all_correct else "ERRORS FOUND"}')

    # Now set random init weights for actual training
    init_trained_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        w = model.get_layer(lname).get_weights()
        w[0][0][0] = random.uniform(0.0, 1.0)
        model.get_layer(lname).set_weights(w)
        init_trained_weights.append(float(w[0][0][0]))

    print(f'\ninit weights: {init_trained_weights}')

    progress_weights = [list(init_trained_weights)]
    progress_losses = []

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.MeanSquaredError()

    for epoch in range(epochs):
        loss = tf.Variable(0.0, name='loss')
        with tf.GradientTape() as tape:
            for line in lines:
                sp = line.split(',')
                xs, ys, true_score = sp[0], sp[1], int(sp[2])
                inp = transform_seqs_to_input(xs, ys)
                inp = transform_input_for_generate(inp)
                inp = tf.constant([inp], dtype=tf.float32)
                logit = model(inp, training=True)
                loss = loss + loss_fn(true_score, logit)
            batch_loss = loss / len(lines)
            grads = tape.gradient(batch_loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

        progress_losses.append(float(batch_loss.numpy()))
        cur = []
        for i in range(LEN):
            lname = 'for_gen_dense_' + str(i + 1)
            cur.append(float(model.get_layer(lname).get_weights()[0][0][0]))
        progress_weights.append(cur)
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f'epoch {epoch:3d} | loss = {float(batch_loss.numpy()):.6f} | weights = {[round(w, 4) for w in cur]}')

    final_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        final_weights.append(float(model.get_layer(lname).get_weights()[0][0][0]))

    print(f'\ninit   weights: {[round(w, 4) for w in init_trained_weights]}')
    print(f'final  weights: {[round(w, 4) for w in final_weights]}')
    print(f'target weights: {target_weights}')
    max_err = max(abs(f - t) for f, t in zip(final_weights, target_weights))
    print(f'max error: {max_err:.6f} {"PASS" if max_err < 0.05 else "NEEDS MORE EPOCHS" if max_err < 0.2 else "FAIL"}')

    return {
        'desire': desire,
        'target_weights': target_weights,
        'init_weights': init_trained_weights,
        'final_weights': final_weights,
        'progress_weights': progress_weights,
        'progress_losses': progress_losses,
        'model': model,
    }


def plot_training(result, title=''):
    progress_weights = np.array(result['progress_weights'])
    losses = result['progress_losses']
    target = result['target_weights']
    LEN = progress_weights.shape[1]
    epochs = progress_weights.shape[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for w_i in range(LEN):
        axes[0].plot(range(epochs), progress_weights[:, w_i], marker='o', markersize=3,
                     label=f'w_{w_i+1} (target={target[w_i]})', color=color_cycle[w_i])
        axes[0].axhline(target[w_i], color=color_cycle[w_i], linestyle='dotted', alpha=0.5)
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('weight value')
    axes[0].set_title('weight trajectories')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

    axes[1].plot(range(1, len(losses) + 1), losses, marker='o', markersize=3, color='red')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('MSE loss')
    axes[1].set_title('training loss')
    axes[1].set_ylim(bottom=0)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Experiment 1 — DESIRE='11'

Baseline. Both characters are `1`, so target weights are `[1, 1]`.

In [ ]:
result_11 = training(
    desire='11',
    csv_path='/content/thesis-edit-distance-nn/sampledata/desired_length_2_levenshtein_2.csv',
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_11, title="DESIRE='11' | Design B | {0,1} alphabet")

## Experiment 2 — DESIRE='01' (the paper's failure case)

Target weights are `[0, 1]`. In the original paper (Design A), `w_1` had a structurally dead gradient because `y[0] = 0` was multiplied into the chain rule. In Design B, the chain rule is `relu'(w_i)` — no `y[i]` factor. So `w_1` should converge to `0` from its positive initialization, with gradient `= 1` the entire way down.

**This is the definitive test:** if `w_1` converges to `0`, the network has genuinely learned that the first character of Y is `0` — purely from edit-distance supervision, with no Y in the forward pass.

In [ ]:
result_01 = training(
    desire='01',
    csv_path='/content/thesis-edit-distance-nn/sampledata/desired_length_2_levenshtein_3.csv',
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_01, title="DESIRE='01' | Design B | {0,1} alphabet")

## Experiment 3 — DESIRE='01011' (5 characters)

Target weights are `[0, 1, 0, 1, 1]`. Positions 0 and 2 must converge to `0`; positions 1, 3, 4 must converge to `1`. All five weights have live gradients under Design B.

In [ ]:
result_01011 = training(
    desire='01011',
    csv_path=csv_path_01011,
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_01011, title="DESIRE='01011' | Design B | {0,1} alphabet")

## Summary

| Notebook | Design | Alphabet | Trainable layer | Target weights | Outcome |
|---|---|---|---|---|---|
| colab1 (original paper) | A: `relu(w_i * y[i])` | `{0, 1}` | `w_i` scales `y[i]` from input | `[1, 1, ...]` | `y[i]=0` kills gradient → stuck |
| colab3 (SiLU, Leaky ReLU) | A with different activations | `{0, 1}` | same | `[1, 1, ...]` | same failure (structural, not activation) |
| colab4 (PSO) | A + gradient-free optimizer | `{0, 1}` | same | `[1, 1, ...]` (but 4 corners) | works but slow, ambiguous solution |
| colab5 (relabel) | A: `relu(w_i * y[i])` | `{1, 2}` | `w_i` scales `y[i]` | `[1, 1, ...]` (always identity) | gradient alive, but weights don't encode Y |
| colab6 attempt 1 | B: `relu(w_i)` | `{1, 2}` | `w_i` IS `y[i]` | `[2, 1, ...]` (char values) | `matching_module` breaks for `relu(w_i) > 2` |
| **colab6 (this)** | **B: `relu(w_i)`** | **`{0, 1}`** | **`w_i` IS `y[i]`** | **`[0, 1, ...]` (char values)** | **expected: works** |

If this experiment succeeds, the contribution is:

1. **Root-cause diagnosis:** the paper's "local solution problem" is a chain-rule artifact from `y[i]` appearing as a multiplicative factor in the gradient.
2. **Architectural fix:** replace `relu(w_i * y[i])` with `relu(w_i)` — remove Y from the forward pass entirely. One-line change.
3. **Result:** weights converge to the actual character values of Y. The network genuinely learns the target string from distance supervision alone. No PSO, no relabeling, no special activations. Vanilla SGD, original alphabet.